# Qwen3.5-9B 导出后 Calibration + LLM Compressor AWQ 量化流程

本 Notebook 用于：
1. 配置环境（uv）
2. 从 SFT 数据生成 calibration.jsonl
3. 用 llm-compressor 对导出模型执行 AWQ W4A16 量化
4. 用 vLLM 以 OpenAI API 格式启动并测试

## 0) 路径约定

请先确认以下路径：
- 已导出的合并模型目录：./export/qwen3.5-9b-ft-merged
- 训练集文件（json 或 jsonl）：例如 ./data/weclone_sft/chat-sft.json
- 根目录已有脚本：build_calibration_jsonl.py

In [ ]:
# 你可以按需修改这些变量
DATASET_PATH = "./data/weclone_sft/chat-sft.json"
MERGED_MODEL_PATH = "./export/qwen3.5-9b-ft-merged"
CALIB_FILE = "./calibration.jsonl"
AWQ_MODEL_PATH = "./export/qwen3.5-9b-ft-merged-awq-ct"
NUM_CALIBRATION_SAMPLES = 256
MAX_CALIB_SEQ_LEN = 512

print('DATASET_PATH =', DATASET_PATH)
print('MERGED_MODEL_PATH =', MERGED_MODEL_PATH)
print('CALIB_FILE =', CALIB_FILE)
print('AWQ_MODEL_PATH =', AWQ_MODEL_PATH)
print('NUM_CALIBRATION_SAMPLES =', NUM_CALIBRATION_SAMPLES)
print('MAX_CALIB_SEQ_LEN =', MAX_CALIB_SEQ_LEN)

## 1) 配置环境（uv）

按你要求，使用 uv pip install --python /path/to/venv/bin/python 语法。

如果你在 Colab / Linux：
- 先创建 venv：uv venv /path/to/venv
- 再安装依赖：uv pip install --python /path/to/venv/bin/python ...

In [ ]:
!uv --version
!uv venv /content/awq-venv
!uv pip install --python /content/awq-venv/bin/python -U llmcompressor transformers datasets accelerate safetensors vllm

## 2) 生成 calibration.jsonl

建议先用 200 条快速验证流程，再提升到 512 或 1000。

In [ ]:
!python build_calibration_jsonl.py \
  --input {DATASET_PATH} \
  --output {CALIB_FILE} \
  --num-samples 512

!head -n 3 {CALIB_FILE}

## 3) LLM Compressor AWQ 量化（W4A16）

默认参数：
- NUM_CALIBRATION_SAMPLES=256
- MAX_CALIB_SEQ_LEN=512
- recipe: AWQModifier(scheme=W4A16, targets=Linear)

说明：
- 这条路线不再依赖 AutoAWQ。
- 会输出 compressed-tensors 格式权重，供 vLLM 加载。

In [ ]:
import json
from pathlib import Path

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

calib_path = Path(CALIB_FILE)
assert calib_path.exists(), f"Calibration file not found: {calib_path}"

texts = []
with calib_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        t = obj.get("text", "")
        if isinstance(t, str) and t.strip():
            texts.append(t.strip())
        if len(texts) >= NUM_CALIBRATION_SAMPLES:
            break

if len(texts) == 0:
    raise RuntimeError("No valid text found in calibration file.")

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_MODEL_PATH,
    trust_remote_code=True,
    use_fast=False,
 )

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_CALIB_SEQ_LEN,
        padding=False,
        add_special_tokens=False,
    )

ds = Dataset.from_dict({"text": texts})
ds = ds.map(tokenize_fn, batched=True, remove_columns=ds.column_names)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype="auto",
    trust_remote_code=True,
 )

recipe = [
    AWQModifier(
        ignore=["lm_head"],
        scheme="W4A16",
        targets=["Linear"],
        duo_scaling=False,
    ),
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_CALIB_SEQ_LEN,
    num_calibration_samples=min(NUM_CALIBRATION_SAMPLES, len(texts)),
)

model.save_pretrained(AWQ_MODEL_PATH, save_compressed=True)
tokenizer.save_pretrained(AWQ_MODEL_PATH)
print("Quantized model saved to:", AWQ_MODEL_PATH)

## 4) 启动 vLLM（OpenAI API）

启动后接口地址：http://127.0.0.1:8000/v1

说明：
- llm-compressor 导出的模型建议使用 --quantization compressed-tensors。

In [ ]:
!vllm serve {AWQ_MODEL_PATH} \
  --quantization compressed-tensors \
  --dtype float16 \
  --served-model-name qwen35-9b-ft-awq \
  --host 0.0.0.0 \
  --port 8000 \
  --max-model-len 1024 \
  --gpu-memory-utilization 0.90

## 5) OpenAI API 格式测试

请在 vLLM 服务启动后执行该单元。

In [ ]:
!curl http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen35-9b-ft-awq","messages":[{"role":"user","content":"你好，做个自我介绍"}],"temperature":0.7}'

## 6) 常见问题

- 如果量化阶段显存不足：把 NUM_CALIBRATION_SAMPLES 从 256 先降到 128，再试 64。
- 如果量化阶段 OOM：把 MAX_CALIB_SEQ_LEN 从 512 再降到 384 或 256。
- 如果 vLLM 启动 OOM：把 --max-model-len 从 1024 再降到 768 或 512。
- 如果 vLLM 提示量化格式不匹配：确认启动参数使用 --quantization compressed-tensors。